# Tema 1 EMQA — notebook de pornire
**Prognoza prețului PZU și volatilitatea, pentru biroul de ofertare al unei companii românești** (cursurile 1–3).

Citiți întîi enunțul (`A1_brief_RO.pdf`). Acest notebook încarcă datele și arată structura pe care trebuie să o urmați. Înlocuiți fiecare `TODO`. Nu schimbați numele obiectelor cerute (`F`, `T5`, `T6`, `T7`, `MY_NUMBERS`): scriptul de verificare al profesorului le citește.

Regulile pe scurt: notebook-ul rulează de la cap la coadă cu *Restart & Run All*; fără date calendaristice introduse direct în cod; nicio informație de după închiderea porții (12:00 CET în ziua d−1); fiecare număr din memoriu provine din acest notebook.

## 0. Parametrii personali

In [ ]:
STUDENT_ID = 'TODO-your-student-ID'   # team: 'ID1+ID2'
MODE = 'personal'                     # the instructor will also run MODE = 'latest' (unseen hold-out)

import os, sys, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))      # folder that contains emqa_assign.py
import emqa_assign as ea
pd.set_option("display.width", 160)

p = ea.params(STUDENT_ID, 1)
if MODE == "latest":
    p = ea.latest_window(p)
print(json.dumps(p, indent=1))
display(ea.data_version())          # rule R3: record the data version

## 1. Date
`P`: prețuri (EUR/MWh), `L`: consum (MW), `R`: eolian + solar (MW); un rînd pe zi de livrare CET, coloanele `h00`…`h23` (ore CET). După 2025-10-01 fiecare valoare orară este media celor patru sferturi de oră.

**Setul de informații (închiderea porții, ora 12:00 CET în ziua *d−1*).**
- prețurile: toate zilele pînă în *d−1* (licitația pentru ziua *d−1* s-a încheiat în *d−2*);
- consumul `L` și producția eoliană+solară `R` realizate: numai pînă în ziua *d−2* (ziua *d−1* nu este încă măsurată); ca substitut (proxy) pentru prognozele OTS folosiți `L`, `R` din ziua *d−2*, la aceeași oră (și *d−3*, *d−7* ca decalaje suplimentare);
- transformările și standardizările: estimate numai pe aceste date.

`ea.leakage_check` verifică această regulă: ascunde `P` începînd cu ziua *d* și `L`, `R` începînd cu ziua *d−1*.

In [ ]:
P, L, R = ea.panel("RO")
t0, t1 = pd.Timestamp(p["test_start"]), pd.Timestamp(p["test_end"])
test = pd.date_range(t0, t1, freq="D")                                         # evaluation days
fdays = pd.date_range(t0 - pd.Timedelta(days=p["burn_in_days"]), t1, freq="D")  # burn-in + test: forecast ALL of these
print(P.shape, P.index.min().date(), "->", P.index.max().date())
print("test:", test[0].date(), "->", test[-1].date(), "| forecast days incl. burn-in:", len(fdays))
display(P.loc[test[:3]].round(1))

## Sarcina 1 — Fereastra de test (tabelul T1 + o figură)

In [ ]:
# TODO: compute on the test window: mean, std, min, max, number of negative hours,
#       number of negative MTUs (rows of ea.quarter_hours("RO"): hourly before 2025-10-01, 15-min after), share of days with 15-min MTU.
T1 = pd.Series(dtype=float)
# TODO: figure: daily base price 2023 -> end of test, test window shaded; y-axis "EUR/MWh".

## Sarcina 2 — Modele de referință naive
Fiecare funcție de prognoză are aceeași semnătură `f(P, L, R, day) -> 24 de numere`, ca `ea.leakage_check` să o poată testa.

In [ ]:
def naive_weron(P, L, R, day):
    """Mon, Sat, Sun: prices of d-7; Tue-Fri: prices of d-1 (same hour)."""
    # TODO
    raise NotImplementedError

def seasonal_naive(P, L, R, day):
    """prices of d-7, same hour"""
    # TODO
    raise NotImplementedError

## Sarcina 3 — Modelul expert ARX, ferestre de calibrare glisante și medierea lor
Scheletul buclei glisante (enunțul, secțiunea *Exemplu rezolvat*, arată o zi calculată de mînă).

In [ ]:
def arx_forecast(P, L, R, day, T):
    """Expert ARX, one regression per hour h, calibrated on the T days d-T..d-1.
    Regressors (transformed prices, see brief): P[d-1,h], P[d-2,h], P[d-7,h], P[d-1,h23],
    min(P[d-1,:]), max(P[d-1,:]), L[d-2,h], R[d-2,h], Monday, Saturday, Sunday dummies, constant.
    L and R: only rows up to d-2 (day d-1 is not yet metered at gate closure), also for their mean and standard deviation."""
    i = P.index.get_loc(day)
    cal = np.arange(i - T, i)          # calibration TARGET days: rows i-T ... i-1 (all before `day`)
    # 1) variance-stabilising transform with median and MAD of the prices in rows (cal[0]-7) ... (i-1)
    # 2) for each hour h: build X_cal (T x 12) and y_cal (T), least squares
    # 3) build x_day from price rows i-1, i-2, i-7 and L/R row i-2 only; predict; back-transform
    # TODO
    raise NotImplementedError

# F = {"Naive (Weron)": DataFrame(fdays x 24), ..., "ARX(56)": ..., "AW-ARX(6 windows)": ...}
F = {}
# TODO: fill F for every model, for all days in fdays
# ea.leakage_check(lambda P, L, R, d: arx_forecast(P, L, R, d, 56), P, L, R, test[0])

## Sarcina 4 — LEAR (autoregresie estimată cu LASSO)

In [ ]:
from sklearn.linear_model import LassoLarsIC
def lear_forecast(P, L, R, day, T=p["lear_window"]):
    """247 regressors: prices d-1, d-2, d-3, d-7 (4 x 24), load and wind+solar proxies of d-2, d-3, d-7
    (2 x 3 x 24), 7 day-of-week dummies. One LassoLarsIC(criterion='aic') per hour."""
    # TODO
    raise NotImplementedError

## Sarcina 5 — Acuratețe și testele Diebold–Mariano
Tabele cerute: `T5` (un rînd pe model) și două matrice de valori p DM.

In [ ]:
T5 = pd.DataFrame(columns=["MAE", "RMSE", "rMAE", "MAE_focus", "RMSE_focus", "rMAE_focus"])   # + per-year columns
# TODO: per calendar year and per quarter; DM (multivariate, |e|, Newey-West variance with 7 lags)

## Sarcina 6 — Prognoze probabilistice (HS și CP)

In [ ]:
QS = np.arange(1, 100) / 100          # the 99 percentiles
# Q[model] : array (n_test_days, 24, 99); errors of the previous p["error_window"] days, same hour
T6 = pd.DataFrame(columns=["APS", "APS_focus", "cov50", "cov90", "cov90_focus", "Kupiec90_focus_p", "width90_focus"])
# TODO

## Sarcina 7 — Volatilitate (GARCH, GJR-GARCH, EGARCH cu erori Student-t)

In [ ]:
from arch import arch_model
daily = ea.daily_prices()["RO"].dropna()
# r_t = 100 * (asinh(P_t) - asinh(P_{t-1}));  remove day-of-week means estimated on the estimation sample only
# estimation sample: [test_start - p["garch_years"] years, test_start);  test: 1-day-ahead variance forecasts
T7 = pd.DataFrame()
# TODO

## Sarcina 8 — Numerele pentru memoriu și tabelul de reconciliere manuală
Orice număr citat în memoriu trebuie să existe în `MY_NUMBERS` (regula R5).

In [ ]:
MY_NUMBERS = {"params": p}
# TODO: add T1, T5, DM, T6, T7 tables (as dicts), the reconciliation rows for p["check_days"] x p["check_hour"],
#       and your decision.
with open(f"my_numbers_A1_{p['student_id']}_{MODE}.json", "w") as fh:
    json.dump(MY_NUMBERS, fh, indent=1, default=str)

## Autoverificare înainte de predare

In [ ]:
checks = {
    "forecasts exist for every model": len(F) >= 10,
    "forecasts cover burn-in + test": all(f.index.equals(fdays) for f in F.values()) if F else False,
    "no missing forecasts": all(not f.isna().any().any() for f in F.values()) if F else False,
    "T5 filled": len(T5) > 0, "T6 filled": len(T6) > 0, "T7 filled": len(T7) > 0,
}
for k, v in checks.items():
    print(("OK   " if v else "FAIL ") + k)